# Baseline — TF-IDF + Linear SVM

Fits `src.config.BASELINE_REGISTRY["tfidf_svm"]`.

## Section 1 — Setup & Data Loading

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import ParameterGrid, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from tqdm.auto import tqdm

sys.path.insert(0, str(Path("../..").resolve()))
from src.config import get_baseline_config
from src.data import load_label_map, load_splits
from src.evaluate import (
    classification_report_text, evaluate_predictions, full_test_report, linear_model_size,
)
from src.plotting import plot_confusion_matrix
from src.runtime import SEED, set_all_seeds

set_all_seeds()

<project>/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
cfg = get_baseline_config("tfidf_svm")
results_dir = cfg.results_dir()

df = load_splits()
LABELS = load_label_map()["labels"]

train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]
test_df = df[df["split"] == "test"]

print(f"split sizes: {df['split'].value_counts().to_dict()}")
print(f"labels ({len(LABELS)}): {LABELS}")

split sizes: {'train': 7614, 'test': 1523, 'val': 1016}
labels (10): ['APELAÇÃO IMPROCEDENTE', 'CONFIRMADA', 'CONFIRMADA A DECISÃO RECORRIDA', 'CONFIRMAÇÃO', 'IMPROCEDENTE', 'NÃO PROVIDO', 'PARCIALMENTE PROCEDENTE', 'PROCEDENTE', 'REVOGADA', 'REVOGADA EM PARTE']


## Section 2 — Grid Search

`PARAM_GRID` spans `tfidf__max_features`, `tfidf__ngram_range`, `clf__penalty` and `clf__C`.

In [3]:
PARAM_GRID = {
    "tfidf__max_features": [10000, 20000, 50000],
    "tfidf__ngram_range": [(1, 2), (1, 3), (1, 4)],
    "clf__penalty": ["l2", "l1"],
    "clf__C": [0.01, 0.03, 0.1, 0.3, 1, 3],
}
N_CV_FOLDS = 5

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(min_df=2, sublinear_tf=True)),
    ("clf", LinearSVC(random_state=SEED, max_iter=5000, **cfg.classifier_kwargs)),
])
cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=SEED)

grid_rows = []
for params in tqdm(list(ParameterGrid(PARAM_GRID)), desc="grid search"):
    pipeline.set_params(**params)
    scores = cross_val_score(
        pipeline, train_df["Sumario"], train_df["label_id"],
        scoring=cfg.scoring, cv=cv, n_jobs=-1,
    )
    row = {k: (list(v) if isinstance(v, tuple) else v) for k, v in params.items()}
    row["mean_cv_macro_f1"] = scores.mean()
    row["std_cv_macro_f1"] = scores.std()
    grid_rows.append(row)

grid_results_df = pd.DataFrame(grid_rows).sort_values("mean_cv_macro_f1", ascending=False).reset_index(drop=True)
grid_results_df.head(10)

grid search:  92%|█████████▏| 99/108 [08:02<00:52,  5.86s/it]<project>/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
<project>/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
grid search:  93%|█████████▎| 100/108 [08:41<02:08, 16.02s/it]<project>/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
grid search:  94%|█████████▎| 101/108 [09:21<02:42, 23.22s/it]<project>/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
grid search:  94%|█████████▍| 102/108 [10:03<02:52, 28.83s/it]<project>/.venv/lib/python3.13/site-packages/sklearn/svm/

,clf__C,clf__penalty,tfidf__max_features,tfidf__ngram_range,mean_cv_macro_f1,std_cv_macro_f1
0,1.0,l2,50000,"[1, 4]",0.327208,0.010261
1,0.3,l2,50000,"[1, 4]",0.326933,0.010098
2,0.3,l2,20000,"[1, 4]",0.324805,0.014892
3,0.3,l2,50000,"[1, 3]",0.322231,0.013939
4,1.0,l2,50000,"[1, 3]",0.321151,0.011820
5,1.0,l2,20000,"[1, 4]",0.321104,0.014059
6,0.1,l2,50000,"[1, 4]",0.319540,0.015792
7,0.3,l2,20000,"[1, 3]",0.318620,0.015633
8,3.0,l2,50000,"[1, 4]",0.317524,0.010116
9,0.1,l2,20000,"[1, 4]",0.317522,0.016220


## Section 3 — Refit on the Best Combination

In [4]:
best_row = grid_results_df.iloc[0]
def _native(key, value):
    if key.endswith("ngram_range"):
        return tuple(value)
    return value.item() if hasattr(value, "item") else value

best_params = {k: _native(k, best_row[k]) for k in PARAM_GRID}

pipeline.set_params(**best_params)
pipeline.fit(train_df["Sumario"], train_df["label_id"])

print(f"best params: {best_params}")
print(f"mean CV macro-F1: {best_row['mean_cv_macro_f1']:.4f}")

best params: {'tfidf__max_features': 50000, 'tfidf__ngram_range': (1, 4), 'clf__penalty': 'l2', 'clf__C': 1.0}
mean CV macro-F1: 0.3272


## Section 4 — Validation Check

`val` played no role in the grid search or refit above.

In [5]:
val_pred_ids = pipeline.predict(val_df["Sumario"]).tolist()
val_metrics = evaluate_predictions(val_df["label_id"].tolist(), val_pred_ids, LABELS)
val_metrics

{'accuracy': 0.39960629921259844,
 'macro_f1': 0.301289457362809,
 'macro_f1_minority': 0.2539098826578059,
 'micro_f1': 0.39960629921259844}

## Section 5 — Final Test Evaluation & Export

Saves the fitted pipeline (`model.joblib`), `metrics.json`.

In [6]:
import json

import joblib

test_pred_ids = pipeline.predict(test_df["Sumario"]).tolist()
test_label_ids = test_df["label_id"].tolist()
test_metrics = evaluate_predictions(test_label_ids, test_pred_ids, LABELS)
test_report = full_test_report(test_label_ids, test_pred_ids, LABELS)
model_size = linear_model_size(pipeline.named_steps["tfidf"], pipeline.named_steps["clf"])

joblib.dump(pipeline, results_dir / "model.joblib")

metrics_payload = {
    "param_grid": {k: (list(map(list, v)) if k.endswith("ngram_range") else v) for k, v in PARAM_GRID.items()},
    "cv_folds": N_CV_FOLDS,
    "grid_results": grid_rows,
    "best_params": best_params,
    "val": val_metrics,
    "test": test_metrics,
    "model_size": model_size,
    "classification_report": test_report["classification_report"],
    "confusion_matrix": test_report["confusion_matrix"],
    "confusion_matrix_labels": test_report["confusion_matrix_labels"],
}
with open(results_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

with open(results_dir / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(classification_report_text(test_label_ids, test_pred_ids, LABELS))

plot_confusion_matrix(
    test_report["confusion_matrix"], LABELS, results_dir / "confusion_matrix.png",
    title=f"{cfg.display_name} — {results_dir.name}",
)

print(f"model size: {model_size}")
test_metrics

model size: {'vocabulary_size': 50000, 'total_parameters': 500010}


{'accuracy': 0.42810242941562704,
 'macro_f1': 0.3466029992607826,
 'macro_f1_minority': 0.3123850283396124,
 'micro_f1': 0.42810242941562704}